# Custom CNN: ảnh đi qua từng lớp

Notebook lấy một ảnh **bị phân loại sai** trong `predictions/test.csv`, chạy forward
qua Custom CNN đã train (`rq1_customcnn_s42`) và hiển thị feature map của mọi lớp:
8 lớp conv, 4 lớp MaxPool, Global Average Pooling và logits.

- Logic thu activation và vẽ hình nằm trong `src/training/feature_maps.py`.
- Ảnh dùng transform validation: `Resize(256)` + `CenterCrop(224)` + normalize ImageNet.
- Model ở `eval()` nên kết quả tất định, không có dropout.


In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT))

from src.data.labels import CANONICAL_CLASSES
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD, build_transforms
from src.models import build_model
from src.training.checkpoint import load_best
from src.training.feature_maps import (
    CUSTOM_CNN_CONV_LAYERS,
    CUSTOM_CNN_POOL_LAYERS,
    collect_activations,
    plot_layer_maps,
    plot_logits,
)
from src.utils.paths import data_root, experiments_root

print(f"Repo: {ROOT}")


## 1. Chọn ảnh bị phân loại sai

Ưu tiên ảnh sai có confidence cao nhất ở lớp dự đoán, vì đây là trường hợp mô hình
"tự tin nhưng sai" — hữu ích nhất khi phân tích lỗi.


In [ ]:
RUN_ID = "rq1_customcnn_s42"
run_dir = experiments_root() / "rq1" / RUN_ID
run = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
class_names = list(run["config"]["data"].get("class_names") or CANONICAL_CLASSES)

predictions = pd.read_csv(run_dir / "predictions" / "test.csv")
probability_columns = [column for column in predictions.columns if column.startswith("prob_")]
wrong = predictions[predictions["correct"] == 0].copy()
wrong["confidence"] = wrong[probability_columns].max(axis=1)
sample = wrong.sort_values("confidence", ascending=False).iloc[0]

image_path = data_root() / sample["path"]
image = Image.open(image_path).convert("RGB")

print(f"Ảnh: {sample['path']}")
print(f"Nhãn thật: {sample['true_label']}")
print(f"Dự đoán:   {sample['pred_label']}")
print(f"Độ tin cậy: {sample['confidence']:.3f}")
print(f"Tổng số ảnh sai: {len(wrong)}/{len(predictions)}")

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title(f"Thật: {sample['true_label']} — Dự đoán: {sample['pred_label']}")
plt.show()


## 2. Dựng lại model và kiểm tra dự đoán

Dùng đúng cấu hình trong `run.json` và checkpoint `best.pt`, sau đó chạy lại trên
đúng tensor đã transform để bảo đảm feature map khớp với lúc đánh giá.


In [ ]:
model_config = dict(run["config"]["model"])
model_name = model_config.pop("name")
num_classes = int(model_config.pop("num_classes", len(class_names)))
model = build_model(model_name, num_classes=num_classes, pretrained=False, **model_config)
load_best(run_dir / "checkpoints" / "best.pt", model, map_location="cpu")
model.eval()

image_size = int(run["config"]["data"].get("image_size", 224))
transform = build_transforms(train=False, image_size=image_size)
tensor = transform(image).unsqueeze(0)

with torch.no_grad():
    logits = model(tensor)
probabilities = torch.softmax(logits, dim=1)[0]
predicted_index = int(probabilities.argmax())

print(f"Dự đoán lại: {class_names[predicted_index]} (p={probabilities[predicted_index]:.3f})")


def denormalize(batch: torch.Tensor) -> "torch.Tensor":
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (batch[0] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[0].set_title("Ảnh gốc")
axes[0].axis("off")
axes[1].imshow(denormalize(tensor))
axes[1].set_title(f"Input model 224×224 - class {class_names[predicted_index]}")
axes[1].axis("off")
fig.tight_layout()
plt.show()


## 3. Thu activation của mọi lớp

Thứ tự layer lấy từ `CUSTOM_CNN_LAYERS`: 8 conv, 4 MaxPool, GAP và classifier.
Kích thước không gian giảm 224 → 112 → 56 → 28 → 14 → 1, số kênh tăng 32 → 256.


In [ ]:
activations = collect_activations(model, tensor)

for name, activation in activations.items():
    print(f"{name:18s} {tuple(activation.shape[1:])}")


## 4. Feature map của 8 lớp conv

Mỗi hàng là một lớp; 6 cột đầu là 6 kênh có phương sai cao nhất, cột cuối là trung
bình trên toàn bộ kênh.


In [ ]:
conv_maps = {name: activations[name] for name in CUSTOM_CNN_CONV_LAYERS}
figure = plot_layer_maps(
    conv_maps,
    max_channels=6,
    title="Custom CNN — feature map của 8 lớp conv",
)
plt.show()


## 5. Sau MaxPool và Global Average Pooling

Các lớp pool giữ nguyên số kênh nhưng giảm một nửa kích thước. GAP cho vector
256 chiều nên được vẽ thành dải màu theo kênh.


In [ ]:
pool_maps = {name: activations[name] for name in CUSTOM_CNN_POOL_LAYERS}
figure = plot_layer_maps(
    pool_maps,
    max_channels=6,
    title="Sau MaxPool và GAP",
)
plt.show()


## 6. Logits và xác suất

Đây là đầu ra cuối cùng trước khi lấy `argmax`. Cột xanh lá là lớp được chọn.


In [ ]:
figure = plot_logits(
    logits,
    class_names,
    title="Logits của ảnh bị phân loại sai",
)
plt.show()


## 7. Nhận xét

- **Stem và các stage đầu** còn giữ chi tiết không gian: kích thước lớn, nhiều kênh
  phản ứng với cạnh, texture và màu.
- **Càng vào sâu**, bản đồ càng nhỏ và thưa; mỗi kênh phản ứng với vùng ngữ nghĩa
  rộng hơn thay vì chi tiết điểm ảnh.
- Với ảnh bị phân loại sai, nên so sánh map của lớp dự đoán và lớp thật: nếu mô hình
  tập trung vào nền hoặc vùng ngoài vật thể, đó là dấu hiệu học shortcut.
- Một kênh riêng lẻ không mang nghĩa "cạnh" hay "texture" một cách chắc chắn; chỉ nên
  đọc xu hướng chung của nhiều kênh và map trung bình.
- Kết quả trên chỉ là **một ảnh minh hoạ**, không dùng để kết luận thống kê.
